In [3]:
import pandas as pd
import re
import unicodedata

### 데이터 기초통계

In [4]:
df_inven = pd.read_csv("../data/raw/maple_inven_questions_원본.csv")

# 결측치 확인
print('질문 데이터 크기 :', df_inven.shape)
print("\n[상위5개 데이터 확인]")
display(df_inven.head(5))
print("\n[구조와 결측 확인]")
display(df_inven.info())
print("\n[카테고리 분포 확인(아이템, 기타, 직업, 시세, 몬스터, 퀘스트)]")
display(df_inven['category'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/maple_inven_questions_원본.csv'

#### 랜덤으로 게시글 제목, 내용 뽑아서 확인

In [ ]:
for i in [0, 3, 8, 10]:
    print(f"[{i}] 게시글 제목 : {df_inven.loc[i, 'title']}")
    print(f"[{i}] 게시글 내용 : {df_inven.loc[i, 'content']}")

[0] 게시글 제목 : 질문 게시글 삭제 및 수정에 대한 이용 안내
[0] 게시글 내용 : 안녕하세요,
인벤
입니다.
커뮤니티 이용 과정에서 질문 게시글을 통해 도움을 주고받는 문화는 인벤의 중요한 소통 방식 중 하나라고 생각합니다. 실제로 많은 회원분들께서 질문을 남기고, 또 다른 회원분들이 자신의 경험과 시간을 들여 정성껏 답변을 남겨주고 계십니다.
다만 최근, 질문에 대한 답변이 등록된 이후 게시글 자체를 삭제하거나 없는 내용으로 수정하는 사례가 반복적으로 확인되고 있습니다. 글을 삭제하게 되는 개인적인 사유는 충분히 있을 수 있으며, 그 이유 자체를 문제 삼고자 하는 것은 아닙니다.
문제는 이러한 활동이 반복될 경우, 성의껏 답변을 남겨준 회원분들께서 허탈감을 느낄 수 있고, 이는 커뮤니티 전반의 질문·답변 문화에도 부정적인 영향을 줄 수 있다는 점입니다. 관련해서 다수의 문의도 접수되었으며, 인벤팀 역시 이러한 말씀에 공감하기도 합니다.
이후, 메이플스토리 인벤 게시판에서 질문 글을 반복적으로 삭제 및 수정하는 행위가 확인될 경우, 게시판 이용에 대한 경고 또는 이용 제한 등의 조치가 이루어질 수 있음을 안내드립니다. 문제가 되는 글이나 유저를 보신다면 신고하기 또는 '인벤운영팀'으로 제보 부탁드립니다.
본 안내는 제한 및 제재를 목적으로 하기보다는, 보다 건강하고 지속적인 커뮤니티 소통 환경을 함께 만들어가기 위한 취지임을 양해 부탁드립니다. 질문 게시글 작성 시에는 해당 점을 한 번 더 고려해 주시길 부탁드립니다.
감사합니다.
[3] 게시글 제목 : 자석펫 확률업 공지 하고 하나요??
[3] 게시글 내용 : 자석펫 사야 하는데
혹시 내일 패치하면 들어올까 해서
공지 하고 하나요??
공지 안 하고 한다면 내일 들어올 가능성 혹시 있을까요?
얼마 전에 한 걸로 아는데 6월쯤인가
[8] 게시글 제목 : 1석->2석 순수자석범위는 같나요?
[8] 게시글 내용 : 순수 자석범위는 3석부터 늘어나는거죠?
2석은 안먹어지던 4층이 먹어지는건 아니고
두마리가 좌

### 불필요한 컬럼 제거
- 남길 컬럼
    - category
    - title
    - created_at
    - views
    - likes
    - content 
    - comment_count
    - comments 
- 버릴 컬럼
    - url 
    - author 
    - crawled_at

In [ ]:
# 원본 데이터는 두고 전처리용 데이터를 복사해서 사용
inven_analysis = df_inven.copy()
# 원본 데이터는 두고 rag용 문서 복사
inven_rag = df_inven.copy()

# 분석에 필요없다고 판단한 컬럼 삭제
inven_analysis = inven_analysis.drop(columns=["author", "crawled_at", "url", "comments"])
display(inven_analysis.info())
# rag 문서에 필요없다고 판단한 컬럼 삭제
inven_rag = inven_rag.drop(columns=["author", "crawled_at"])
display(inven_rag.info())

<class 'pandas.DataFrame'>
RangeIndex: 4244 entries, 0 to 4243
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   category       4244 non-null   str  
 1   title          4244 non-null   str  
 2   created_at     4244 non-null   str  
 3   views          4244 non-null   int64
 4   likes          4244 non-null   int64
 5   content        4234 non-null   str  
 6   comment_count  4244 non-null   int64
dtypes: int64(3), str(4)
memory usage: 232.2 KB


None

<class 'pandas.DataFrame'>
RangeIndex: 4244 entries, 0 to 4243
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   url            4244 non-null   str  
 1   category       4244 non-null   str  
 2   title          4244 non-null   str  
 3   created_at     4244 non-null   str  
 4   views          4244 non-null   int64
 5   likes          4244 non-null   int64
 6   content        4234 non-null   str  
 7   comment_count  4244 non-null   int64
 8   comments       4244 non-null   str  
dtypes: int64(3), str(6)
memory usage: 298.5 KB


None

### 형식 맞추기
#### created_at 컬럼
시간은 필요 없다고 판단

In [ ]:
# 날짜 형식 맞추기(시간은 삭제)
inven_analysis['created_at'] = pd.to_datetime(inven_analysis['created_at']).dt.strftime('%Y-%m-%d')
display(inven_analysis['created_at'].head(5))

0    2026-01-22
1    2026-08-19
2    2026-08-19
3    2026-08-19
4    2026-08-19
Name: created_at, dtype: str

### 결측치 제거
#### content 컬럼
게시글 내용이 없으므로 불필요하다고 판단

In [ ]:
inven_analysis = inven_analysis.dropna(subset=["content"])
display(inven_analysis.info())

<class 'pandas.DataFrame'>
Index: 4234 entries, 0 to 4243
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   category       4234 non-null   str  
 1   title          4234 non-null   str  
 2   created_at     4234 non-null   str  
 3   views          4234 non-null   int64
 4   likes          4234 non-null   int64
 5   content        4234 non-null   str  
 6   comment_count  4234 non-null   int64
dtypes: int64(3), str(4)
memory usage: 264.6 KB


None

### HTML/마크업 제거, URL 제거 또는 치환, 공백 정규화, 특수문자 정규화, 반복 문자 정규화, 토큰 길이 또는 텍스트 길이 필터링

In [ ]:
# title
inven_analysis["title_clean"] = inven_analysis["title"].apply(
    lambda x: re.sub(r"\s+", " ",
                re.sub(r"[\n\r\t]", " ",
                re.sub(r"\\n|\\r|\\t", " ", 
                re.sub(r"(.)\1{2,}", r"\1\1\1",
                re.sub(r"^\[[^\]]+\]\s*", "", x),
                ),
            ),
        ),
    ).strip()
)

# content
inven_analysis["content_clean"] = inven_analysis["content"].apply(
    lambda x: re.sub(r"(.)\1{1,}", r"\1\1",
                re.sub(r"\s+", " ",
                re.sub(r"https?://\S+|www\.\S+", " ",
                re.sub(r"[\n\r\t]", " ",
                re.sub(r"\\n|\\r|\\t", " ",
                re.sub(r"^\[[^\]]+\]\s*", "", x)
                    )
                )
            )
        )
    ).strip()
)

# 본문에서 비정상적으로 조합된 한글 발견 -- 제거
inven_analysis['title_clean'] = inven_analysis['title_clean'].apply(lambda x: unicodedata.normalize('NFC', x))
inven_analysis['content_clean'] = inven_analysis['content_clean'].fillna('').apply(lambda x: unicodedata.normalize('NFC', x))

# 관리자 공자사항 행 삭제
no_analysis = ['질문 게시글 삭제 및 수정에 대한 이용 안내']
inven_analysis = inven_analysis[~inven_analysis['title_clean'].isin(no_analysis)].copy()
print('관리자 공지 제거 후 데이터 수:', len(inven_analysis))

# 분석을 위해 title과 content를 합침
inven_analysis['analysis_text'] = (inven_analysis['title_clean'].fillna('') + ' ' 
                                   + inven_analysis['content_clean'].fillna('')).str.strip()
print('분석을 위해 title과 content를 합침')
display(inven_analysis[['title_clean', 'content_clean', 'analysis_text']])

# 질문이 명확하지 않은 글만 후보로 뽑기
# 질문 또는 도움 요청에서 자주 나타나는 표현
question_pattern = (
    r'\?|'
    r'어떻게|왜|뭐|무엇|어디|'
    r'가능|되나요|될까요|인가요|맞나요|있나요|'
    r'질문|문의|궁금|'
    r'추천|부탁|알려|'
    r'고민|조언|도와'
)

# 질문 표현이 하나라도 있는지 확인
inven_analysis['has_question_signal'] = (
    inven_analysis['analysis_text']
    .str.contains(question_pattern, regex=True, na=False)
)
print('질문/비질문 분류')
inven_analysis['has_question_signal'].value_counts()

관리자 공지 제거 후 데이터 수: 4233
분석을 위해 title과 content를 합침


,title_clean,content_clean,analysis_text
1,모멘텀패스 살까요 플러스 나오는 거 살까요?,챌섭에서 렌 키우고 있습니다. 지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던전...,모멘텀패스 살까요 플러스 나오는 거 살까요? 챌섭에서 렌 키우고 있습니다. 지금 렙...
2,인벤 계정 레벨 어떻게 올리는거에요?,오메나.. 10렙을 목표로하고 있는데 이거 쉽지 않은거 맞지요?,인벤 계정 레벨 어떻게 올리는거에요? 오메나.. 10렙을 목표로하고 있는데 이거 쉽...
3,자석펫 확률업 공지 하고 하나요??,자석펫 사야 하는데 혹시 내일 패치하면 들어올까 해서 공지 하고 하나요?? 공지 안...,자석펫 확률업 공지 하고 하나요?? 자석펫 사야 하는데 혹시 내일 패치하면 들어올까...
4,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,받기전 3배 이벤창에서밖에 교환못함요..?,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요?? 받기전 3배 이벤창에서밖에...
5,뉴비 템세팅 질문...,템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 앞으로 템세팅 어떻게 해...,뉴비 템세팅 질문... 템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 ...
...,...,...,...
4239,투표 박아줘~~~!!♡!@@♡!♡!@!@♡!,"렌나올때 시작해서 어느정도 정착은 했는데, 이번 챌섭에서 열심히 키워서 둘이서 상위...",투표 박아줘~~~!!♡!@@♡!♡!@!@♡! 렌나올때 시작해서 어느정도 정착은 했는...
4240,찐막 직업질문,챌섭 팬텀할까요 아크할까요 마음은 팬텀인데 아크도 너무 멋있어보여서,찐막 직업질문 챌섭 팬텀할까요 아크할까요 마음은 팬텀인데 아크도 너무 멋있어보여서
4241,렌 나왔을때 잠깐했는데 복귀해도 같은 루트인가요?,렌 나왔을때 제네패스 사고 흔히 이야기하는 검밑솔 좀 잡고 이지 칼로스인가 까지 잡...,렌 나왔을때 잠깐했는데 복귀해도 같은 루트인가요? 렌 나왔을때 제네패스 사고 흔히 ...
4242,챌섭 드메 100 100 가격 질문,"정말 귀찮으시겠지만, 제가 드메템을 맞춰본 적이 없어서요 ㅠㅠ 보통 드메 100 1...","챌섭 드메 100 100 가격 질문 정말 귀찮으시겠지만, 제가 드메템을 맞춰본 적이..."


질문/비질문 분류


has_question_signal
True     4077
False     156
Name: count, dtype: int64

In [ ]:
question_idx = [
    73, 79, 96, 104, 346, 354, 297, 1045, 1039, 1013,
    1002, 969, 937, 832, 827, 798, 1076, 1095, 1139, 1166,
    1262, 1477, 1515, 1832, 1911, 2121, 2134, 2196, 2215,
    2254, 2301, 2419, 2503, 2511, 2522, 2553, 2682, 2738,
    2750, 2786, 2806, 2829, 2830, 2849, 2867,
    2928, 2935, 2950, 2958, 3027, 3123, 3128, 3144, 3178,
    3219, 3279, 3352, 3409, 3439, 3508, 3583, 3592, 3602,
    3608, 3634, 3692, 3735, 3787, 3812, 3874, 3875, 3882,
    3956, 4003, 4019, 4046, 4053, 4094, 4108, 4109, 4147
]

non_question_idx = [
    2, 131, 145, 166, 195, 197, 214, 224, 240, 263, 273,
    371, 384, 452, 470, 478, 511, 548, 601, 626, 645,
    723, 776, 962, 1047, 1053, 1056, 1067, 1115, 1225,
    1499, 1519, 1683, 1696, 1844, 1848, 1857, 1878, 1885,
    1899, 1943, 2077, 2202, 2239, 2290, 2346, 2434, 2485,
    2527, 2613, 2700, 2797, 2852, 2880,
    2884, 2955, 3091, 3180, 3195, 3255, 3345, 3372, 3462,
    3564, 3609, 3668, 3687, 3894, 3961, 3980, 3982, 3992,
    4049, 4209, 146, 3897, 
]

inven_analysis['is_question'] = True
inven_analysis.loc[non_question_idx, 'is_question'] = False

print(inven_analysis['is_question'].value_counts())

is_question
True     4157
False      76
Name: count, dtype: int64


In [ ]:
# 기초 전처리 + 질문 검수 결과 저장
inven_analysis.to_csv(
    "../data/processed/inven_analysis_v1.csv", encoding="utf-8", index=False
)
inven_analysis_v1 = pd.read_csv("../data/processed/inven_analysis_v1.csv")

print("질문 데이터 크기 :", inven_analysis_v1.shape)
print("\n[상위5개 데이터 확인]")
display(inven_analysis_v1.head(5))
print("\n[구조와 결측 확인]")
display(inven_analysis_v1.info())
print("\n[카테고리 분포 확인(아이템, 기타, 직업, 시세, 몬스터, 퀘스트)]")
display(inven_analysis_v1["category"].value_counts())

질문 데이터 크기 : (4233, 12)

[상위5개 데이터 확인]


,category,title,created_at,views,likes,content,comment_count,title_clean,content_clean,analysis_text,has_question_signal,is_question
0,기타,모멘텀패스 살까요 플러스 나오는 거 살까요?,2026-08-19,36,0,챌섭에서 렌 키우고 있습니다.\n지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던...,1,모멘텀패스 살까요 플러스 나오는 거 살까요?,챌섭에서 렌 키우고 있습니다. 지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던전...,모멘텀패스 살까요 플러스 나오는 거 살까요? 챌섭에서 렌 키우고 있습니다. 지금 렙...,True,True
1,기타,인벤 계정 레벨 어떻게 올리는거에요?,2026-08-19,28,0,오메나... 10렙을 목표로하고 있는데 이거 쉽지 않은거 맞지요?,0,인벤 계정 레벨 어떻게 올리는거에요?,오메나.. 10렙을 목표로하고 있는데 이거 쉽지 않은거 맞지요?,인벤 계정 레벨 어떻게 올리는거에요? 오메나.. 10렙을 목표로하고 있는데 이거 쉽...,True,False
2,기타,자석펫 확률업 공지 하고 하나요??,2026-08-19,143,0,자석펫 사야 하는데\n혹시 내일 패치하면 들어올까 해서\n공지 하고 하나요??\n공...,0,자석펫 확률업 공지 하고 하나요??,자석펫 사야 하는데 혹시 내일 패치하면 들어올까 해서 공지 하고 하나요?? 공지 안...,자석펫 확률업 공지 하고 하나요?? 자석펫 사야 하는데 혹시 내일 패치하면 들어올까...,True,True
3,아이템,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,2026-08-19,157,0,받기전 3배 이벤창에서밖에 교환못함요...?,2,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,받기전 3배 이벤창에서밖에 교환못함요..?,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요?? 받기전 3배 이벤창에서밖에...,True,True
4,아이템,뉴비 템세팅 질문.....,2026-08-19,166,0,템환 6.7 헥환 5.7 찍힙니다\n익스우 솔플정도 목표인데 앞으로 템세팅 어떻게 ...,4,뉴비 템세팅 질문...,템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 앞으로 템세팅 어떻게 해...,뉴비 템세팅 질문... 템환 6.7 헥환 5.7 찍힙니다 익스우 솔플정도 목표인데 ...,True,True



[구조와 결측 확인]
<class 'pandas.DataFrame'>
RangeIndex: 4233 entries, 0 to 4232
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   category             4233 non-null   str  
 1   title                4233 non-null   str  
 2   created_at           4233 non-null   str  
 3   views                4233 non-null   int64
 4   likes                4233 non-null   int64
 5   content              4233 non-null   str  
 6   comment_count        4233 non-null   int64
 7   title_clean          4233 non-null   str  
 8   content_clean        4232 non-null   str  
 9   analysis_text        4233 non-null   str  
 10  has_question_signal  4233 non-null   bool 
 11  is_question          4233 non-null   bool 
dtypes: bool(2), int64(3), str(7)
memory usage: 339.1 KB


None


[카테고리 분포 확인(아이템, 기타, 직업, 시세, 몬스터, 퀘스트)]


category
아이템    2060
기타     1360
직업      394
시세      161
몬스터     147
퀘스트     111
Name: count, dtype: int64

In [ ]:
# is_question=True 만 따로 저장한 csv파일
inven_question = inven_analysis[inven_analysis["is_question"]].copy()
inven_question.to_csv(
    "../data/processed/inven_question_final.csv", index=False, encoding="utf-8"
)

print("[최종 데이터 크기]")
print(inven_question.shape)
print("\n[결측치]")
print(inven_question.isna().sum())
print("\n[전체 행 중복]")
print(inven_question.duplicated().sum())
print("\n[analysis_text 중복]")
print(inven_question["analysis_text"].duplicated().sum())
print("\n[빈 analysis_text]")
print(
    (inven_question["analysis_text"].str.strip() == "").sum()
)
print("\n[카테고리 분포]")
print(inven_question["category"].value_counts())
print("\n[수집 기간]")
print(inven_question["created_at"].min())
print(inven_question["created_at"].max())

# handoff_cols = [
#     "category",
#     "title",
#     "created_at",
#     "views",
#     "likes",
#     "content",
#     "comment_count",
#     "title_clean",
#     "content_clean",
#     "analysis_text"
# ]

# inven_question_final = inven_question[handoff_cols].copy()
# inven_question_final.to_csv(
#     "../data/processed/inven_question_final.csv",
#     index=False,
#     encoding="utf-8"
# )

NameError: name 'inven_analysis' is not defined